In [ ]:
%%writefile eikonale.h
#pragma once           // il file viene incluso una sola volta
// Risolutore locale 3D, comune al FIM a nodi e al Block FIM
#include <vector>
#include <cmath>      // fminf, sqrtf
const float INF = 1e30f;                 // "infinito": nodo non ancora raggiunto dall'onda

// Risolutore locale 3D (Godunov, Alg. 2 esteso al 3D): dai minimi dei vicini a, b, c al nuovo u
// Prova con 1, 2, poi 3 vicini; usato sia dal FIM a nodi sia dal Block FIM (GPU)
__host__ __device__ float solve_quadratic3(float a, float b, float c,float F, float h)
{
    float Fh = F * h;                              // tempo per attraversare una cella
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}

In [ ]:
%%writefile ambiente.h
// Contiene: i parametri, gli ostacoli (scatole, sfere, fori), la maschera della griglia,
// le sorgenti della passata 1 (bordo degli ostacoli)
#pragma once           // il file viene incluso una sola volta
#include <vector>
#include <cmath>
#include <algorithm>

// Forme con cui si descrive l'ambiente (coordinate nel cubo [-1, 1]^3)
struct Punto   { double x, y, z; };
struct Scatola { double x0, x1, y0, y1, z0, z1; };       // parallelepipedo: x0 <= x <= x1, y0 <= y <= y1, z0 <= z <= z1
struct Sfera   { double cx, cy, cz, r; };                 // centro (cx, cy, cz) e raggio r


// AMBIENTE: per cambiarlo si modifica solo questa parte
// =====================================================================
const double MARGINE = 0.06;                             // distanza minima dagli ostacoli (raggio del drone + sicurezza)
const double D_SICUREZZA = 0.3;                          // distanza dagli ostacoli oltre la quale si va a velocita' piena
const double V_MIN = 0.1;                                // velocita' minima, a ridosso degli ostacoli (> 0)

const Punto A = { -0.85, -0.85, -0.85 };                 // partenza del drone: angolo in basso
const Punto B = {  0.85,  0.85,  0.85 };                 // arrivo (sorgente del FIM): angolo opposto, in alto

const std::vector<Scatola> SCATOLE = {                   // {x0, x1, y0, y1, z0, z1}
    { -0.45, -0.35, -1.0, 1.0, -1.0, 1.0 },              // muro 1 (x = -0.4)
    {  0.15,  0.25, -1.0, 1.0, -1.0, 1.0 },              // muro 2 (x = 0.2)
    {  0.45,  0.65,  0.2, 1.0,  0.2, 1.0 },              // blocco davanti a B: si entra solo da sotto o di lato
    { -1.0,  -0.45, -0.3, -0.2, -1.0, 0.4 },             // paratia tra A e il muro 1
    { -1.0,  -0.7,  -1.0, -0.3, -0.2, -0.1 },            // mensola sopra A: si sale dal varco vicino al muro 1
    { -0.15, -0.05, -0.05, 0.05, -1.0, 1.0 },            // pilastro tra i due muri, da terra al soffitto
    {  0.25,  1.0,  -0.45, -0.35, 0.3, 1.0 },            // trave appesa al soffitto dopo il muro 2: si passa sotto
};
const std::vector<Sfera> SFERE = {                       // {cx, cy, cz, r}
    { -0.1,  0.3, -0.4, 0.25 },                          // sfera tra i due muri, davanti alla finestra 1
    { -0.1, -0.4,  0.4, 0.22 },                          // sfera tra i due muri, davanti alla finestra 2
    {  0.6, -0.1, -0.1, 0.2 },                           // sfera dopo il muro 2, sotto la trave
};
const std::vector<Scatola> FORI = {                      // zone in cui l'ostacolo viene tolto
    { -0.45, -0.35,  0.5,  0.8, -0.8, -0.5 },            // finestra nel muro 1: y alta, z bassa
    {  0.15,  0.25, -0.8, -0.5,  0.5,  0.8 },            // finestra nel muro 2: y bassa, z alta
};
// =====================================================================

// Vero se il punto (x, y, z) e' dentro la scatola s (bordi compresi)
inline bool dentro(const Scatola& s, double x, double y, double z)
{
    return s.x0 <= x && x <= s.x1 && s.y0 <= y && y <= s.y1 && s.z0 <= z && z <= s.z1;
}

// Vero se il punto (x, y, z) e' dentro la sfera s (distanza dal centro <= raggio)
inline bool dentro(const Sfera& s, double x, double y, double z)
{
    return (x - s.cx) * (x - s.cx) + (y - s.cy) * (y - s.cy) + (z - s.cz) * (z - s.cz) <= s.r * s.r;
}

// Vero se il punto (x, y, z) e' dentro un ostacolo (i FORI tolgono l'ostacolo)
inline bool ostacolo(double x, double y, double z)
{
    for (const auto& f : FORI)    if (dentro(f, x, y, z)) return false;
    for (const auto& s : SCATOLE) if (dentro(s, x, y, z)) return true;
    for (const auto& s : SFERE)   if (dentro(s, x, y, z)) return true;
    return false;
}

// Maschera della griglia: 1 = libero, 0 = ostacolo (resta INF)
// Gli ostacoli vengono allargati di MARGINE (ingombro del drone)
inline std::vector<unsigned char> costruisci_maschera(int N)
{
    const double h = 2.0 / (N - 1);
    std::vector<unsigned char> mask((size_t)N * N * N);
    for (int k = 0; k < N; ++k)                              // 1) gli ostacoli veri
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j)
                mask[((size_t)k * N + i) * N + j] = ostacolo(-1.0 + j * h, -1.0 + i * h, -1.0 + k * h) ? 0 : 1;

    // 2) Margine: diventa ostacolo ogni nodo entro MARGINE da un ostacolo
    const int r = (int)std::ceil(MARGINE / h);               // raggio del margine, in nodi
    std::vector<unsigned char> allargata = mask;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
                if (mask[((size_t)k * N + i) * N + j]) continue;                       // si parte solo dai nodi ostacolo...
                bool bordo = false;                        // ...che toccano un nodo libero
                if (j > 0     && mask[((size_t)k * N + i) * N + j - 1]) bordo = true;
                if (j < N - 1 && mask[((size_t)k * N + i) * N + j + 1]) bordo = true;
                if (i > 0     && mask[((size_t)k * N + i - 1) * N + j]) bordo = true;
                if (i < N - 1 && mask[((size_t)k * N + i + 1) * N + j]) bordo = true;
                if (k > 0     && mask[((size_t)(k - 1) * N + i) * N + j]) bordo = true;
                if (k < N - 1 && mask[((size_t)(k + 1) * N + i) * N + j]) bordo = true;
                if (!bordo) continue;
                for (int dk = -r; dk <= r; ++dk)                  // tutti i nodi entro la sfera di raggio MARGINE
                    for (int di = -r; di <= r; ++di)
                        for (int dj = -r; dj <= r; ++dj) {
                            const int kk = k + dk, ii = i + di, jj = j + dj;
                            if (kk < 0 || kk >= N || ii < 0 || ii >= N || jj < 0 || jj >= N) continue;
                            if ((dk * dk + di * di + dj * dj) * h * h <= MARGINE * MARGINE)
                                allargata[((size_t)kk * N + ii) * N + jj] = 0;
                        }
            }
    return allargata;
}

// Sorgenti della passata 1: i nodi liberi che toccano un ostacolo
inline std::vector<int> nodi_di_bordo(const std::vector<unsigned char>& mask, int N)
{
    std::vector<int> bordo;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
                const int q = (k * N + i) * N + j;
                if (!mask[q]) continue;                                  // solo nodi liberi...
                if ((j > 0 && !mask[q - 1]) || (j < N - 1 && !mask[q + 1]) ||
                    (i > 0 && !mask[q - N]) || (i < N - 1 && !mask[q + N]) ||
                    (k > 0 && !mask[q - N * N]) || (k < N - 1 && !mask[q + N * N]))
                    bordo.push_back(q);                                  // ...che hanno un vicino ostacolo
            }
    return bordo;
}

// FM2: dalla distanza d dagli ostacoli alla lentezza F = 1/v della passata 2
// v = V_MIN + (1 - V_MIN) * min(d / D_SICUREZZA, 1): F = 10 sul bordo, 1 lontano
inline std::vector<float> lentezza_fm2(const std::vector<float>& d)
{
    std::vector<float> F(d.size());
    for (size_t q = 0; q < d.size(); ++q) {
        const double r = std::min((double)d[q] / D_SICUREZZA, 1.0);       // distanza normalizzata: 0 sul bordo, 1 oltre D_SICUREZZA (o se d = INF)
        F[q] = (float)(1.0 / (V_MIN + (1.0 - V_MIN) * r));
    }
    return F;
}

// Controllo iniziale: A e B non devono stare dentro un ostacolo
inline bool ambiente_valido()
{
    return !ostacolo(A.x, A.y, A.z) && !ostacolo(B.x, B.y, B.z);
}

// Dal punto (x, y, z) all'indice del nodo piu' vicino (serve per A e B)
int nodo(int N, double x, double y, double z){
    const double h = 2.0 / (N - 1);
    const int j = (int)lround((x + 1.0) / h), i = (int)lround((y + 1.0) / h), k = (int)lround((z + 1.0) / h);
    return (k * N + i) * N + j;
}

In [ ]:
%%writefile block_fim.cuh
#pragma once           // il file viene incluso una sola volta
#include <cstdio>
#include <cstdlib>
#include <vector>
#include <algorithm>       // std::min
#include <cuda_runtime.h>
#include "eikonale.h"      // INF e solve_quadratic3

// Controllo degli errori CUDA: se una chiamata fallisce, stampa dove e perche' ed esce
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

#define BLOCK 8  // blocco 8 x 8 x 8 = 512 nodi, un thread per nodo

// Risolutore locale in shared memory: i 6 vicini del nodo sono nella copia s (blocco + halo)
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}


// Step 1: niter iterazioni; Step 2: 1 iterazione di prova.
__device__ void process_block(int block, int niter, float* u, const float* F, const unsigned char* mask,
                             int N, int nbx, float h, float eps, unsigned char* Cb_out)
{
    __shared__ float s_u[BLOCK + 2][BLOCK + 2][BLOCK + 2];                // blocco + halo (10^3 valori), nella shared memory
    __shared__ unsigned char s_c[BLOCK * BLOCK * BLOCK];                  // i flag Cn dei 512 nodi

    int bi = (block / nbx) % nbx, bj = block % nbx, bk = block / (nbx * nbx);            // riga, colonna e strato del blocco
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    int tid = (tz * BLOCK + ty) * BLOCK + tx;                                       // numero del thread nel blocco: 0..511
    int i = bi * BLOCK + ty, j = bj * BLOCK + tx, k = bk * BLOCK + tz;              // nodo di questo thread
    int idx = (k * N + i) * N + j;                                                  // posizione del nodo nella griglia
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || !mask[idx];             // nodo da non aggiornare: fuori dalla griglia o ostacolo
    float f = inside ? F[idx] : 1.0f;                  // F_ijk del nodo

    // 1) Global -> Shared: 1000 celle (blocco + halo), 512 thread
    for (int m = tid; m < (BLOCK + 2) * (BLOCK + 2) * (BLOCK + 2); m += BLOCK * BLOCK * BLOCK) {
        int sz = m / ((BLOCK + 2) * (BLOCK + 2));
        int sy = (m / (BLOCK + 2)) % (BLOCK + 2);
        int sx = m % (BLOCK + 2);                             // cella (sz, sy, sx) della shared memory
        int gi = bi * BLOCK + sy - 1;                         // nodo della griglia corrispondente:
        int gj = bj * BLOCK + sx - 1;                         
        int gk = bk * BLOCK + sz - 1;                         // -1 perche' lo strato/riga/colonna 0 e' l'halo
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();                                         // la copia e' completa prima di usarla

    // 2) niter iterazioni locali in shared memory
    bool conv = true;                                        // Cn del nodo
    for (int it = 0; it < niter; ++it) {
        float p = s_u[tz+1][ty + 1][tx + 1];                 // valore prima dell'aggiornamento
        float q = fixed ? p                                  // fuori dominio: non cambia
                        : solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);
        __syncthreads();                                     // (A) tutti hanno letto i vicini...
        s_u[tz+1][ty + 1][tx + 1] = q;
        __syncthreads();                                     // (B) ...e tutti hanno scritto
        conv = fabsf(p - q) < eps;                           // Cn: il nodo non e' cambiato
    }

    // 3) Shared -> Global: ogni thread riscrive il proprio nodo (non l'halo)
    if (inside) u[idx] = s_u[tz+1][ty + 1][tx + 1];

    // 4) Cb(b) <- reduction(Cn(b)): AND dei 512 flag
    s_c[tid] = conv;
    __syncthreads();
    for (int stride = BLOCK * BLOCK * BLOCK / 2; stride > 0; stride /= 2) {     // 256, ..., 4, 2, 1
        if (tid < stride) s_c[tid] = s_c[tid] && s_c[tid + stride];
        __syncthreads();
    }
    if (tid == 0) Cb_out[block] = s_c[0];                     // vero solo se TUTTI i nodi sono convergenti
}

// Step 1 (Alg. 7): ogni blocco della lista A fa niter iterazioni
__global__ void update_active_blocks(const int* list, float* u, const float* F, const unsigned char* mask,
                                    int N, int nbx, float h, int niter,
                                    float eps, unsigned char* Cb1)
{
    process_block(list[blockIdx.x], niter, u, F, mask, N, nbx, h, eps, Cb1);
}

// Inizio di ogni giro: costruisce la lista A dei blocchi attivi (Cb = 0)
__global__ void build_active_list(const unsigned char* Cb, unsigned char* active,
                                  int* list, int* count, int n_blocks) 
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;           // numero del blocco di nodi
    if (t >= n_blocks) return;                               // thread in eccesso
    bool act = !Cb[t];                                       // attivo se NON convergente
    active[t] = act;
    if (act) list[atomicAdd(count, 1)] = t;                  // aggiungi t alla lista A
}

// Step 2, parte 1: i 6 vicini dei blocchi appena convergenti diventano candidati
__global__ void build_candidate_list(const int* list, int count, int nbx, const unsigned char* active,
                                     const unsigned char* Cb1, unsigned char* Cb,
                                     int* cand, int* cand_list, int* cand_count)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;           // blockDim = 256
    if (t >= count) return;                                  // thread in eccesso
    int b = list[t];                                         // un blocco della lista A
    Cb[b] = Cb1[b];                                          // non convergente (Caso 1): resta in A
    if (!Cb1[b]) return;                                     // proseguono solo i blocchi appena convergenti

    int bk = b / (nbx * nbx), bi = (b / nbx) % nbx, bj = b % nbx;                          // strato, riga e colonna del blocco
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};  // sinistra, destra, sotto, sopra e davanti, dietro
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // fuori dalla griglia di blocchi
        if (active[nb[d]] && !Cb1[nb[d]]) continue;          // Caso 1: lo aggiorna già lo Step 1
        if (atomicExch(&cand[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}

// Step 2, parte 2: ogni candidato fa 1 iterazione; se cambia, diventa attivo
__global__ void update_candidate_blocks(const int* cand_list, const int* cand_count, int* cand,
                                        float* u, const float* F, const unsigned char* mask, int N, int nbx, float h, float eps, unsigned char* Cb)
{
    if (blockIdx.x >= *cand_count) return;                   // blocco CUDA in eccesso
    int b = cand_list[blockIdx.x];
    process_block(b, 1, u, F, mask, N, nbx, h, eps, Cb);      // test del blocco: 1 iterazione
    if (threadIdx.x == 0 && threadIdx.y == 0 && threadIdx.z == 0) cand[b] = 0;   // pronto per il giro successivo
}

// Block FIM (Alg. 7) sulla GPU: restituisce u, il tempo del ciclo (ms) e il numero di giri
// Ogni giro: lista A -> Step 1 -> Step 2; fine quando A e' vuota
int block_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
              const std::vector<int>& srcs, const std::vector<unsigned char>& mask, float eps, int niter, float* ms)
{
    const int nbx = (N + BLOCK - 1) / BLOCK, n_blocks = nbx * nbx * nbx; // blocchi per lato e blocchi totali
    const size_t bytes = (size_t)N * N * N * sizeof(float); // dimensione in byte della griglia

    std::vector<unsigned char> Cb0(n_blocks, 1);    // flag Cb iniziali: 1 = convergente, 0 = attivo
    // I blocchi che contengono una sorgente partono attivi Cb0 = 0
    for (int s : srcs) {
        const int sk = s / (N * N), si = (s / N) % N, sj = s % N;   // strato, riga e colonna del nodo sorgente
        Cb0[((sk / BLOCK) * nbx + si / BLOCK) * nbx + sj / BLOCK] = 0;   // il blocco di questa sorgente parte attivo
    }

    // Memoria sulla GPU e copia dei dati iniziali (CPU -> GPU)
    float *d_u, *d_F; unsigned char *d_Cb, *d_Cb1, *d_active; int *d_list, *d_count;
    int *d_cand, *d_cand_list, *d_cand_count; unsigned char* d_mask;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_mask, (size_t)N * N * N));
    CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_Cb1, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_active, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_count, sizeof(int)));

    // Memoria per i blocchi candidati
    CUDA_CHECK(cudaMalloc(&d_cand, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));

    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_mask, mask.data(), (size_t)N * N * N, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb0.data(), n_blocks, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_Cb1, 0, n_blocks));
    CUDA_CHECK(cudaMemset(d_cand, 0, n_blocks * sizeof(int)));


    const int threads = 256, list_blocks = (n_blocks + threads - 1) / threads;   // kernel "di servizio": 256 thread, un thread per blocco di nodi
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                      // "segna l'ora" prima del ciclo (si misura solo il ciclo)
    int iter = 0;
    while (true) {                                           // un giro dell'Alg. 7
        int count = 0;
        CUDA_CHECK(cudaMemset(d_count, 0, sizeof(int)));
        build_active_list<<<list_blocks, threads>>>(d_Cb, d_active, d_list, d_count, n_blocks);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemcpy(&count, d_count, sizeof(int), cudaMemcpyDeviceToHost));
        if (count == 0) break;  // se count == 0, lista A vuota, termina ciclo
        ++iter;
        // Step 1: foreach b in A in parallelo
        update_active_blocks<<<count, dim3(BLOCK, BLOCK, BLOCK)>>>(d_list, d_u, d_F, d_mask, N, nbx, h, niter, eps, d_Cb1);    // si lanciano count blocchi CUDA, ognuno con 512 thread
        CUDA_CHECK(cudaGetLastError());
        // Step 2: solo i vicini dei blocchi appena convergenti (i candidati)
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));
        build_candidate_list<<<(count + threads - 1) / threads, threads>>>(d_list, count, nbx, d_active, d_Cb1, d_Cb,
                                                                          d_cand, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());
        update_candidate_blocks<<<std::min(6 * count, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, d_cand,
                                                                                    d_u, d_F, d_mask, N, nbx, h, eps, d_Cb);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    // Risultato sulla CPU (GPU -> CPU) e liberazione della memoria
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));CUDA_CHECK(cudaFree(d_Cb));
    CUDA_CHECK(cudaFree(d_Cb1)); CUDA_CHECK(cudaFree(d_mask));
    CUDA_CHECK(cudaFree(d_active)); CUDA_CHECK(cudaFree(d_list)); CUDA_CHECK(cudaFree(d_count));
    CUDA_CHECK(cudaFree(d_cand)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    return iter;
}

In [ ]:
%%writefile node_fim.cuh
#pragma once           // il file viene incluso una sola volta
// FIM a nodi (Alg. 6 della tesi) sulla GPU: lista attiva di nodi, un thread per nodo, niente shared memory
// Versione con maschera (ostacoli fermi a INF) e piu' sorgenti
#include "block_fim.cuh"   // CUDA_CHECK, INF e solve_quadratic3

// Nuovo u del nodo x, leggendo i 6 vicini dalla memoria globale (INF fuori dalla griglia)
__device__ float solve_global(const float* u, const float* F, int x, int N, float h)
{
    int k = x / (N * N), i = (x / N) % N, j = x % N;             // strato, riga e colonna del nodo
    float a = fminf(j > 0 ? u[x - 1] : INF, j < N - 1 ? u[x + 1] : INF);              // vicini lungo x
    float b = fminf(i > 0 ? u[x - N] : INF, i < N - 1 ? u[x + N] : INF);              // vicini lungo y
    float c = fminf(k > 0 ? u[x - N * N] : INF, k < N - 1 ? u[x + N * N] : INF);      // vicini lungo z
    return fminf(u[x], solve_quadratic3(a, b, c, F[x], h));      // return min(u_x, u)
}

// Alg. 6, prima parte: ogni nodo x della lista A si aggiorna (p <- u(x), q <- g(x), u(x) <- q)
__global__ void node_update(const int* list, int count, float* u, const float* F,
                            int N, float h, float eps, unsigned char* conv)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;               // un thread per nodo della lista
    if (t >= count) return;
    int x = list[t];
    float p = u[x];
    float q = solve_global(u, F, x, N, h);
    u[x] = q;
    conv[t] = fabsf(p - q) < eps;                                // |p - q| < eps: nodo convergente
}

// Alg. 6, seconda parte: un nodo non convergente resta in A; uno convergente esce
// e prova i vicini fuori da A: se un vicino migliora, entra nella nuova lista
__global__ void node_neighbors(const int* list, int count, const unsigned char* conv,
                               const unsigned char* in_list, const unsigned char* mask, float* u, const float* F, int N, float h,
                               int* next_flag, int* next_list, int* next_count)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;
    if (t >= count) return;
    int x = list[t];
    if (!conv[t]) {                                              // non convergente: resta attivo
        if (atomicExch(&next_flag[x], 1) == 0) next_list[atomicAdd(next_count, 1)] = x;
        return;
    }
    int k = x / (N * N), i = (x / N) % N, j = x % N;
    int  nb[6] = { x - 1,  x + 1,     x - N, x + N,     x - N * N, x + N * N };   // sinistra, destra, sotto, sopra, davanti, dietro
    bool ok[6] = { j > 0, j < N - 1, i > 0, i < N - 1, k > 0,     k < N - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d] || in_list[nb[d]] || !mask[nb[d]]) continue;  // fuori griglia, gia' in A, oppure ostacolo
        int y = nb[d];
        float p = u[y];
        float q = solve_global(u, F, y, N, h);
        if (q < p) {                                             // p > q: il vicino migliora
            u[y] = q;
            if (atomicExch(&next_flag[y], 1) == 0)               // aggiungi a A (una volta sola)
                next_list[atomicAdd(next_count, 1)] = y;
        }
    }
}

// Aggiornamento dei flag di appartenenza: la lista vecchia esce
__global__ void clear_in_list(const int* list, int count, unsigned char* in_list)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;
    if (t < count) in_list[list[t]] = 0;
}

// e la nuova entra (next_flag torna a 0 per l'iterazione successiva)
__global__ void set_in_list(const int* list, int count, unsigned char* in_list, int* next_flag)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;
    if (t < count) { in_list[list[t]] = 1; next_flag[list[t]] = 0; }
}

// FIM a nodi (Alg. 6) sulla GPU: restituisce u, il tempo del ciclo (ms) e il numero di iterazioni
int node_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
             const std::vector<int>& srcs, const std::vector<unsigned char>& mask, float eps, float* ms)
{
    const int n = N * N * N;                                     // numero di nodi
    const size_t bytes = (size_t)n * sizeof(float);
    const int threads = 256;

    // Memoria sulla GPU
    float *d_u, *d_F; int *d_list, *d_next, *d_count, *d_flag; unsigned char *d_conv, *d_in, *d_mask;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_list, n * sizeof(int)));            // lista A attuale
    CUDA_CHECK(cudaMalloc(&d_next, n * sizeof(int)));            // lista A del passo successivo
    CUDA_CHECK(cudaMalloc(&d_count, sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_flag, n * sizeof(int)));            // 1 = gia' messo nella nuova lista
    CUDA_CHECK(cudaMalloc(&d_conv, n));                          // convergenza dei nodi della lista
    CUDA_CHECK(cudaMalloc(&d_in, n));                            // 1 = nodo in A
    CUDA_CHECK(cudaMalloc(&d_mask, n));                          // 1 = libero, 0 = ostacolo
    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_mask, mask.data(), n, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_flag, 0, n * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_in, 0, n));

    // All'inizio A contiene le sorgenti
    int count = (int)srcs.size();
    CUDA_CHECK(cudaMemcpy(d_list, srcs.data(), count * sizeof(int), cudaMemcpyHostToDevice));
    set_in_list<<<(count + threads - 1) / threads, threads>>>(d_list, count, d_in, d_flag);
    CUDA_CHECK(cudaGetLastError());

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                          // si misura solo il ciclo
    int iter = 0;
    while (count > 0) {                                          // while A != vuoto
        ++iter;
        const int blocks = (count + threads - 1) / threads;
        node_update<<<blocks, threads>>>(d_list, count, d_u, d_F, N, h, eps, d_conv);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemset(d_count, 0, sizeof(int)));
        node_neighbors<<<blocks, threads>>>(d_list, count, d_conv, d_in, d_mask, d_u, d_F, N, h,
                                            d_flag, d_next, d_count);
        CUDA_CHECK(cudaGetLastError());
        clear_in_list<<<blocks, threads>>>(d_list, count, d_in);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemcpy(&count, d_count, sizeof(int), cudaMemcpyDeviceToHost));   // dimensione della nuova A
        if (count > 0) {
            set_in_list<<<(count + threads - 1) / threads, threads>>>(d_next, count, d_in, d_flag);
            CUDA_CHECK(cudaGetLastError());
        }
        std::swap(d_list, d_next);                               // la nuova lista diventa quella attuale
    }
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    // Risultato sulla CPU e pulizia
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));
    CUDA_CHECK(cudaFree(d_list)); CUDA_CHECK(cudaFree(d_next)); CUDA_CHECK(cudaFree(d_count));
    CUDA_CHECK(cudaFree(d_flag)); CUDA_CHECK(cudaFree(d_conv)); CUDA_CHECK(cudaFree(d_in));
    CUDA_CHECK(cudaFree(d_mask));
    return iter;
}

In [ ]:
%%writefile main.cu
// Confronto FIM a nodi (Alg. 6) e Block FIM (Alg. 7), entrambi sulla GPU, nel caso con ostacoli e F non uniforme:
// il FM2 del drone (passata 1 = distanza dagli ostacoli con F = 1, passata 2 = tempo da B con la F del FM2)
#include <cstdio>
#include <algorithm>       // std::min
#include "eikonale.h"      // INF e risolutore locale
#include "ambiente.h"      // ostacoli, maschera, FM2
#include "block_fim.cuh"   // Block FIM (Alg. 7)
#include "node_fim.cuh"    // FIM a nodi (Alg. 6)

int main()
{
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d\n", prop.name, prop.multiProcessorCount);

    CUDA_CHECK(cudaFree(0));                                 // inizializza la GPU prima delle misure
    const float eps = 1e-6f;                                 // tolleranza di convergenza (uguale per i due metodi)
    const int niter = BLOCK;                                 // 8 iterazioni locali del Block FIM (la scelta migliore per il FM2)
    const int ripetizioni = 3;                               // ogni tempo = il minimo di 3 misure

    if (!ambiente_valido()) {                                // A o B dentro un ostacolo: il problema non ha senso
        printf("Errore: A o B sono dentro un ostacolo, correggi ambiente.h\n");
        return 1;
    }
    printf("tempi = passata 1 + passata 2; speedup = t FIM a nodi / t Block FIM\n\n");
    printf("    N   iter. FIM   giri Block FIM   max|FIM - BlockFIM|   u(A) FIM   u(A) Block FIM   t FIM [ms]   t Block FIM [ms]   speedup\n");

    for (int Nk : { 33, 65, 129, 193, 257 }) {
        const float hk = 2.0f / (Nk - 1);                    // passo della griglia nel cubo [-1, 1]^3
        const int n = Nk * Nk * Nk;
        const std::vector<unsigned char> mask = costruisci_maschera(Nk);    // 1 = libero, 0 = ostacolo
        const std::vector<int> srcs = { nodo(Nk, B.x, B.y, B.z) };          // sorgente della passata 2: l'arrivo B
        const int iA = nodo(Nk, A.x, A.y, A.z);              // nodo della partenza A

        // passata 1: distanza dagli ostacoli (sorgenti = bordo degli ostacoli, F = 1)
        const std::vector<int> bordo = nodi_di_bordo(mask, Nk);
        std::vector<float> d0(n, INF);
        for (int s : bordo) d0[s] = 0.0f;
        const std::vector<float> uno(n, 1.0f);
        std::vector<float> d_node, d_block;
        double t1_node = 1e30, t1_block = 1e30;
        for (int r = 0; r < ripetizioni; ++r) {              // 3 misure, si tiene la piu' veloce
            float ms = 0.0f;
            d_node = d0;
            node_fim(d_node, uno, Nk, hk, bordo, mask, eps, &ms);                // GPU: Alg. 6
            t1_node = std::min(t1_node, (double)ms);
            d_block = d0;
            block_fim(d_block, uno, Nk, hk, bordo, mask, eps, niter, &ms);       // GPU: Alg. 7
            t1_block = std::min(t1_block, (double)ms);
        }

        // passata 2: tempo da B con la F del FM2 (la stessa F per i due metodi, ricavata dal Block FIM)
        const std::vector<float> Fk = lentezza_fm2(d_block);
        std::vector<float> u0(n, INF);
        for (int s : srcs) u0[s] = 0.0f;
        std::vector<float> u_node, u_block;
        double t2_node = 1e30, t2_block = 1e30;
        int it_node = 0, it_block = 0;
        for (int r = 0; r < ripetizioni; ++r) {
            float ms = 0.0f;
            u_node = u0;
            it_node = node_fim(u_node, Fk, Nk, hk, srcs, mask, eps, &ms);        // GPU: Alg. 6
            t2_node = std::min(t2_node, (double)ms);
            u_block = u0;
            it_block = block_fim(u_block, Fk, Nk, hk, srcs, mask, eps, niter, &ms);   // GPU: Alg. 7
            t2_block = std::min(t2_block, (double)ms);
        }

        float diff = 0.0f;                                   // i due metodi devono dare lo stesso risultato in tutte e due le passate
        for (int q = 0; q < n; ++q) {
            diff = fmaxf(diff, fabsf(d_node[q] - d_block[q]));
            diff = fmaxf(diff, fabsf(u_node[q] - u_block[q]));
        }
        const double t_node = t1_node + t2_node, t_block = t1_block + t2_block;
        printf("%5d   %9d   %14d   %19.2e   %8.4f   %14.4f   %10.2f   %16.2f   %7.1f\n", Nk, it_node, it_block,
               diff, u_node[iA], u_block[iA], t_node, t_block, t_node / t_block);
    }
    return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 main.cu -o confronto_fm2 && ./confronto_fm2